# HPAFL: Hybrid Privacy-Aware Federated Learning on Kaggle T4 GPU

This notebook implements a complete Hybrid Privacy-Aware Federated Learning (HPAFL) system on the Kaggle T4 GPU. It demonstrates:
- **Federated Learning** with multiple hospital clients
- **Differential Privacy** using Opacus DP-SGD
- **Secure Aggregation** with pairwise masking
- **Adaptive Weighted Aggregation** based on accuracy, data quality, and reliability
- **FedBN** (Batch Normalization not aggregated across hospitals)

**Dataset**: HAM10000 (10,000 skin lesion images, 7 classes)

**GPU**: T4 (16GB VRAM - sufficient for batch_size 64 with DP-SGD)

**Runtime**: ~20 minutes for 2 rounds (demo) or ~35-40 minutes for full training (20 rounds)

---

## ⚠️ IMPORTANT: Dataset Setup

**Before running this notebook, you MUST add the dataset:**

1. Click **Settings** (⚙️) in the top right
2. Under **Data**, click **Add data input**
3. Search for and select: **`skin-cancer-mnist-ham10000`**
4. Click **Add** and wait for the dataset to be mounted
5. Then proceed with running the cells

**Without the dataset**, you'll get a "FileNotFoundError: Metadata CSV not found" error when running the training cells.

## Cell 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision efficientnet_pytorch opacus flwr scipy pillow pydantic fastapi uvicorn streamlit plotly requests scikit-learn pandas numpy matplotlib tqdm

## Cell 2: Setup Environment and Verify GPU

In [ ]:
import os
import sys
from pathlib import Path

# Setup paths
os.chdir('/kaggle/working')
sys.path.insert(0, '/kaggle/working')

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
    print("✓ GPU ready for training!")
else:
    print("✗ GPU not available. CPU training will be very slow.")

## Cell 3: Clone HPAFL Framework

In [ ]:
import subprocess

# Clone the framework
repo_url = "https://github.com/Sadiya-125/COE-Project.git"

# Check if already cloned
if not Path('/kaggle/working/COE-Project').exists():
    print(f"Cloning from {repo_url}...")
    result = subprocess.run(
        ['git', 'clone', repo_url],
        cwd='/kaggle/working',
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Framework cloned successfully")
    else:
        print(f"✗ Clone failed. Please upload files manually.")
        print(f"Error: {result.stderr}")
else:
    print("✓ Framework already exists")

# Change to hpafl-framework directory and add to path
os.chdir('/kaggle/working/COE-Project/hpafl-framework')
sys.path.insert(0, '/kaggle/working/COE-Project/hpafl-framework')
print(f"Working directory: {os.getcwd()}")

## Cell 4: Import HPAFL Modules

In [ ]:
import logging
import json
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger(__name__)

# Import HPAFL modules
from config import HAPFLConfig
from scripts.run_hpafl import run_hpafl

print("✓ HPAFL modules imported successfully")

## Cell 5: Base Configuration (Demo Settings)

In [ ]:
# Create configuration with demo settings
cfg = HAPFLConfig()

# ============ Dataset paths (Kaggle-specific) ============
cfg.data_root = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
cfg.output_root = "/kaggle/working"

# ============ Demo Training Parameters ============
# Note: Demo uses 2 rounds/1 epoch for quick testing (~20 min)
# See strategy cells below to improve accuracy to ~0.80+
cfg.num_rounds = 2
cfg.local_epochs = 1
cfg.batch_size = 64
cfg.num_workers = 4

# ============ Model parameters ============
cfg.model_name = "efficientnet_b0"
cfg.pretrained = True
cfg.image_size = 224

# ============ Privacy parameters (Differential Privacy) ============
cfg.target_epsilon = 8.0
cfg.target_delta = 1e-5
cfg.max_grad_norm = 1.0
cfg.noise_multiplier = 1.1

# ============ Federated parameters ============
cfg.num_hospitals = 3
cfg.fraction_fit = 1.0
cfg.fraction_evaluate = 1.0

# ============ Adaptive aggregation weights ============
cfg.alpha_accuracy = 0.40
cfg.alpha_reliability = 0.30
cfg.alpha_data_quality = 0.20
cfg.alpha_historical = 0.10
cfg.ema_decay = 0.30

# ============ Class imbalance handling ============
cfg.focal_gamma = 2.0
cfg.focal_alpha = 2.0        # Will be overridden by strategy cells if used
cfg.use_augmentation = True   # Will be overridden by strategy cells if used
cfg.augmentation_strength = "high"  # Will be overridden by strategy cells if used

print("="*70)
print("HPAFL Configuration for Kaggle T4 GPU (DEMO SETTINGS)")
print("="*70)
print(f"Data root:              {cfg.data_root}")
print(f"Output root:            {cfg.output_root}")
print(f"\nTraining Settings:")
print(f"  Num rounds:           {cfg.num_rounds}")
print(f"  Local epochs:         {cfg.local_epochs}")
print(f"  Batch size:           {cfg.batch_size}")
print(f"  Num workers:          {cfg.num_workers}")
print(f"\nModel:                  {cfg.model_name}")
print(f"Pretrained:             {cfg.pretrained}")
print(f"\nDifferential Privacy:")
print(f"  Target ε (epsilon):   {cfg.target_epsilon}")
print(f"  Noise multiplier:     {cfg.noise_multiplier}")
print(f"\nClass Imbalance Handling:")
print(f"  Focal Gamma:          {cfg.focal_gamma}")
print(f"  Focal Alpha:          {cfg.focal_alpha}")
print(f"  Data Augmentation:    {cfg.use_augmentation}")
print(f"  Augmentation Strength: {cfg.augmentation_strength}")
print("="*70)
print("\n⚠️  DEMO CONFIGURATION IN USE")
print("Expected accuracy: ~0.68 | Time: ~20 minutes")
print("\nTo improve accuracy to ~0.80+, run ONE of the strategy cells below.")
print("="*70)

## Cell 5.1: Accuracy Optimization Strategies

**Choose ONE strategy below to improve accuracy from 0.68 → 0.80+**

| Strategy | Accuracy | Time | Privacy (ε) | Best For |
|----------|----------|------|------------|----------|
| **Demo** (current) | 0.68 | ~20 min | 7.998 | Quick testing |
| **Strategy 1** | ~0.80 | 35-40 min | 7.80 | Production (RECOMMENDED) |
| **Strategy 2** | ~0.76-0.78 | ~70 min | 3.90 | Research, time-limited |
| **Strategy 3** | ~0.72-0.74 | 35-40 min | 4.0 | HIPAA/GDPR (strict privacy) |
| **Strategy 4** | ~0.82+ | 40-45 min | 7.80 | Max performance (risky) |

**How to use:**
1. Run Cell 5 above to set demo configuration
2. Run **ONE** of the strategy cells below (5.2, 5.3, 5.4, or 5.5)
3. Proceed with Cell 6 (Data Diagnostics) and beyond
4. Skip strategy cells you don't want to use

**Note**: Each strategy cell completely overrides the demo configuration with production settings.

### 5.2: STRATEGY 1 - Full Training (RECOMMENDED)

In [ ]:
# STRATEGY 1: FULL TRAINING (RECOMMENDED)
# Best for: Production use, maximum accuracy (~0.80)
# Time: 35-40 minutes on Kaggle T4
# Privacy: ε ≈ 7.80 (within 8.0 budget)

print("="*70)
print("STRATEGY 1: FULL TRAINING (RECOMMENDED)")
print("="*70)
print("\nApplying production-quality configuration (~0.80 accuracy)...\n")

cfg.num_rounds = 20
cfg.local_epochs = 5
cfg.batch_size = 64
cfg.learning_rate = 1e-3
cfg.max_grad_norm = 1.5
cfg.target_epsilon = 15.0
cfg.noise_multiplier = 0.8
cfg.focal_alpha = 3.0
cfg.use_augmentation = True
cfg.augmentation_strength = "high"

print("✓ Strategy 1 Configuration Applied:")
print(f"  Rounds: {cfg.num_rounds} | Epochs: {cfg.local_epochs} | Batch: {cfg.batch_size}")
print(f"  Learning Rate: {cfg.learning_rate} | Max Grad Norm: {cfg.max_grad_norm}")
print(f"  Target ε (Epsilon): {cfg.target_epsilon} | Noise Multiplier: {cfg.noise_multiplier}")
print(f"  Focal Alpha: {cfg.focal_alpha} | Augmentation: {cfg.augmentation_strength}")
print(f"\n  EXPECTED: ~0.80 accuracy, ~35-40 minutes")
print("="*70)

### 5.3: STRATEGY 2 - Balanced Approach

In [ ]:
# STRATEGY 2: BALANCED APPROACH
# Best for: Research, when time/compute is limited
# Time: ~70 minutes on Kaggle T4
# Privacy: ε ≈ 3.90 (strict but reasonable)

print("="*70)
print("STRATEGY 2: BALANCED APPROACH")
print("="*70)
print("\nApplying balanced configuration (~0.76-0.78 accuracy)...\n")

cfg.num_rounds = 10
cfg.local_epochs = 3
cfg.batch_size = 64
cfg.learning_rate = 5e-4
cfg.max_grad_norm = 1.2
cfg.target_epsilon = 10.0
cfg.noise_multiplier = 1.0
cfg.focal_alpha = 2.5
cfg.use_augmentation = True
cfg.augmentation_strength = "medium"

print("✓ Strategy 2 Configuration Applied:")
print(f"  Rounds: {cfg.num_rounds} | Epochs: {cfg.local_epochs} | Batch: {cfg.batch_size}")
print(f"  Learning Rate: {cfg.learning_rate} | Max Grad Norm: {cfg.max_grad_norm}")
print(f"  Target ε (Epsilon): {cfg.target_epsilon} | Noise Multiplier: {cfg.noise_multiplier}")
print(f"  Focal Alpha: {cfg.focal_alpha} | Augmentation: {cfg.augmentation_strength}")
print(f"\n  EXPECTED: ~0.76-0.78 accuracy, ~70 minutes")
print("="*70)

### 5.4: STRATEGY 3 - Privacy-First (HIPAA/GDPR Compliant)

In [ ]:
# STRATEGY 3: PRIVACY-FIRST (HIPAA/GDPR COMPLIANT)
# Best for: Applications requiring strict differential privacy
# Time: 35-40 minutes on Kaggle T4
# Privacy: ε ≈ 4.0 (HIPAA/GDPR-compliant)

print("="*70)
print("STRATEGY 3: PRIVACY-FIRST (HIPAA/GDPR COMPLIANT)")
print("="*70)
print("\nApplying strict privacy configuration (~0.72-0.74 accuracy)...\n")

cfg.num_rounds = 20
cfg.local_epochs = 5
cfg.batch_size = 64
cfg.learning_rate = 1e-4
cfg.max_grad_norm = 0.8
cfg.target_epsilon = 4.0
cfg.noise_multiplier = 1.5
cfg.focal_alpha = 2.0
cfg.use_augmentation = True
cfg.augmentation_strength = "high"

print("✓ Strategy 3 Configuration Applied:")
print(f"  Rounds: {cfg.num_rounds} | Epochs: {cfg.local_epochs} | Batch: {cfg.batch_size}")
print(f"  Learning Rate: {cfg.learning_rate} | Max Grad Norm: {cfg.max_grad_norm}")
print(f"  Target ε (Epsilon): {cfg.target_epsilon} | Noise Multiplier: {cfg.noise_multiplier}")
print(f"  Focal Alpha: {cfg.focal_alpha} | Augmentation: {cfg.augmentation_strength}")
print(f"\n  EXPECTED: ~0.72-0.74 accuracy, HIPAA/GDPR-compliant privacy (ε≈4.0)")
print("  ⚠️  Privacy guarantee: (ε=4.0, δ=1e-5)-differential privacy")
print("="*70)

### 5.5: STRATEGY 4 - Memory-Intensive (Maximum Performance)

In [ ]:
# STRATEGY 4: MEMORY-INTENSIVE (MAXIMUM PERFORMANCE)
# Best for: Systems with unlimited compute budget
# Time: 40-45 minutes on Kaggle T4
# Privacy: ε ≈ 7.80 (same as Strategy 1)
# ⚠️  WARNING: May exceed Kaggle T4's 16GB VRAM - test first!

print("="*70)
print("STRATEGY 4: MEMORY-INTENSIVE (MAXIMUM PERFORMANCE)")
print("="*70)
print("\n⚠️  WARNING: This strategy may exceed Kaggle T4's 16GB VRAM!")
print("Test memory first. If OOM errors occur, use Strategy 1 instead.\n")

cfg.num_rounds = 20
cfg.local_epochs = 5
cfg.batch_size = 128
cfg.learning_rate = 5e-4
cfg.max_grad_norm = 2.0
cfg.target_epsilon = 15.0
cfg.noise_multiplier = 0.7
cfg.focal_alpha = 4.0
cfg.use_augmentation = True
cfg.augmentation_strength = "high"

print("✓ Strategy 4 Configuration Applied:")
print(f"  Rounds: {cfg.num_rounds} | Epochs: {cfg.local_epochs} | Batch: {cfg.batch_size}")
print(f"  Learning Rate: {cfg.learning_rate} | Max Grad Norm: {cfg.max_grad_norm}")
print(f"  Target ε (Epsilon): {cfg.target_epsilon} | Noise Multiplier: {cfg.noise_multiplier}")
print(f"  Focal Alpha: {cfg.focal_alpha} | Augmentation: {cfg.augmentation_strength}")
print(f"\n  EXPECTED: ~0.82+ accuracy (MAXIMUM)")
print("\n  💾 Memory usage:")
print(f"    Batch size: {cfg.batch_size} (2x normal)")
print(f"    Estimated GPU memory: ~14-15 GB (tight on T4)")
print("="*70)

### 5.6: Configuration Summary

In [ ]:
# Display current configuration (reflects demo or whichever strategy was run)

print("\n" + "="*70)
print("CURRENT HPAFL CONFIGURATION (for training)")
print("="*70)
print("\n📊 TRAINING PARAMETERS:")
print(f"  • Federated Rounds:    {cfg.num_rounds}")
print(f"  • Local Epochs:        {cfg.local_epochs}")
print(f"  • Batch Size:          {cfg.batch_size}")
print(f"  • Learning Rate:       {cfg.learning_rate}")

print("\n🔒 PRIVACY PARAMETERS (Differential Privacy):")
print(f"  • Target ε (epsilon):  {cfg.target_epsilon}")
print(f"  • Noise Multiplier:    {cfg.noise_multiplier}")
print(f"  • Max Grad Norm:       {cfg.max_grad_norm}")

print("\n⚖️  CLASS IMBALANCE HANDLING:")
print(f"  • Focal Loss Gamma:    {cfg.focal_gamma}")
print(f"  • Focal Alpha:         {cfg.focal_alpha}")
print(f"  • Data Augmentation:   {cfg.use_augmentation}")
print(f"  • Augment Strength:    {cfg.augmentation_strength}")

print("\n📁 DATA & I/O:")
print(f"  • Data Root:           {cfg.data_root}")
print(f"  • Output Root:         {cfg.output_root}")
print(f"  • Num Hospitals:       {cfg.num_hospitals}")

print("\n⚙️  INFRASTRUCTURE:")
print(f"  • Num Workers:         {cfg.num_workers}")
print(f"  • Image Size:          {cfg.image_size}×{cfg.image_size}")
print(f"  • Model:               {cfg.model_name} (pretrained={cfg.pretrained})")

print("\n" + "="*70)
print("Ready to train! Proceed to next cells.")
print("="*70 + "\n")

## Cell 6: Data Diagnostics

In [ ]:
from pathlib import Path

print("="*70)
print("DATA DIAGNOSTICS - Checking HAM10000 Dataset")
print("="*70)

data_root = cfg.data_root
print(f"\n1. Configured data_root: {data_root}")
print(f"   Directory exists: {Path(data_root).exists()}")

if Path(data_root).exists():
    print(f"\n2. Contents of {data_root}:")
    contents = list(Path(data_root).iterdir())
    if contents:
        for item in sorted(contents):
            if item.is_file():
                size_mb = item.stat().st_size / 1e6
                print(f"   📄 {item.name:<40} ({size_mb:.1f} MB)")
            else:
                try:
                    item_count = len(list(item.iterdir()))
                    print(f"   📁 {item.name}/ ({item_count} items)")
                except:
                    print(f"   📁 {item.name}/ (access denied)")
    else:
        print("   ⚠️  Directory is empty!")

    # Check for metadata CSV specifically
    csv_path = Path(data_root) / "HAM10000_metadata.csv"
    print(f"\n3. Looking for metadata CSV:")
    print(f"   Expected: {csv_path}")
    print(f"   Found: {csv_path.exists()}")
    
    if csv_path.exists():
        print(f"   ✓ Metadata CSV found!")
        print(f"   Size: {csv_path.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"\n   ❌ Metadata CSV not found!")
        print(f"   Searching for any CSV files...")
        csv_files = list(Path(data_root).glob("**/*.csv"))
        if csv_files:
            print(f"   Found {len(csv_files)} CSV file(s):")
            for csv_file in csv_files:
                rel_path = csv_file.relative_to(data_root)
                print(f"     - {rel_path}")
        else:
            print(f"   No CSV files found in {data_root}")
            print(f"\n   ⚠️  IMPORTANT: Make sure you added the dataset in Kaggle settings:")
            print(f"   1. Go to notebook settings")
            print(f"   2. Under 'Data', add: 'skin-cancer-mnist-ham10000'")
            print(f"   3. Re-run this cell after adding the dataset")
else:
    print(f"\n✗ Data root directory not found: {data_root}")
    print(f"\nPlease verify:")
    print(f"1. Dataset is added in Kaggle notebook settings")
    print(f"2. Dataset name: 'skin-cancer-mnist-ham10000' (by kmader)")
    print(f"3. Working directory is correct: {os.getcwd()}")

print("\n" + "="*70)

## Cell 7: Prepare Data

In [ ]:
import subprocess
import sys

# Prepare data by running the data preparation script
print("="*70)
print("STEP 1: Preparing Data - Creating Hospital Partitions")
print("="*70)

try:
    # Run the data preparation script from hpafl-framework directory
    result = subprocess.run(
        [sys.executable, 'data/prepare_data.py'],
        cwd='/kaggle/working/COE-Project/hpafl-framework',
        capture_output=True,
        text=True,
        timeout=300
    )
    
    print(result.stdout)
    if result.stderr:
        print("Warnings/Info:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ Data preparation completed successfully!")
        print("✓ Hospital partitions created (non-IID Dirichlet splits)")
    else:
        print(f"\n✗ Data preparation failed with return code {result.returncode}")
        print("Error output:")
        print(result.stderr)
        
except subprocess.TimeoutExpired:
    print("✗ Data preparation timed out after 5 minutes")
except Exception as e:
    print(f"✗ Error during data preparation: {e}")
    import traceback
    traceback.print_exc()

# Verify partition files exist
partition_dir = Path('/kaggle/working/COE-Project/hpafl-framework/partitions')
if partition_dir.exists():
    partition_files = list(partition_dir.glob('*.json'))
    print(f"\n✓ Found {len(partition_files)} partition files:")
    for f in sorted(partition_files):
        print(f"  - {f.name}")
else:
    print(f"\n⚠️  Partition directory not found at {partition_dir}")
    print("   Partitions will be created automatically during training if needed.")

## Cell 8: Run HPAFL Training

In [ ]:
# Run the full HPAFL pipeline
try:
    print("\n" + "="*70)
    print("Starting HPAFL Federated Learning Training")
    print("="*70 + "\n")
    
    run_hpafl(cfg)
    
    print("\n" + "="*70)
    print("✓ HPAFL TRAINING COMPLETED SUCCESSFULLY!")
    print("="*70)
    
except Exception as e:
    print(f"\n✗ Error during training: {e}")
    import traceback
    traceback.print_exc()

## Cell 9: Load and Display Results

In [ ]:
import pandas as pd
import json

results_dir = Path(cfg.output_root) / "results"

print("\n" + "="*70)
print("HPAFL RESULTS SUMMARY")
print("="*70)

# 1. Privacy Budget
privacy_file = results_dir / "privacy_budget.csv"
if privacy_file.exists():
    privacy_df = pd.read_csv(privacy_file)
    print("\n📊 Privacy Budget Consumption:")
    print(privacy_df.to_string(index=False))
    print(f"\n  Final ε (epsilon): {privacy_df['epsilon'].iloc[-1]:.4f}")
    print(f"  Target ε: {cfg.target_epsilon:.4f}")
else:
    print("\n⚠️  Privacy budget file not found")

# 2. Adaptive Aggregation Weights
weights_file = results_dir / "adaptive_weights.csv"
if weights_file.exists():
    weights_df = pd.read_csv(weights_file)
    print("\n⚖️  Adaptive Aggregation Weights (by round):")
    for round_num in sorted(weights_df['round'].unique()):
        round_data = weights_df[weights_df['round'] == round_num]
        print(f"\n  Round {round_num}:")
        for _, row in round_data.iterrows():
            print(f"    Hospital {row['hospital_id']}: weight={row['weight']:.4f}, score={row['score']:.4f}")
else:
    print("\n⚠️  Adaptive weights file not found")

# 3. Final Results
results_file = results_dir / "hpafl.json"
if results_file.exists():
    with open(results_file) as f:
        final_results = json.load(f)
    
    print("\n🎯 Final Performance Metrics:")
    print(f"  Global Accuracy:      {final_results.get('global_accuracy', 'N/A'):.4f}")
    print(f"  Global F1 (Macro):    {final_results.get('global_f1_macro', 'N/A'):.4f}")
    print(f"  Total Rounds:         {final_results.get('num_rounds', 'N/A')}")
    print(f"  Final ε (epsilon):    {final_results.get('final_epsilon', 'N/A'):.4f}")
    print(f"  Training Time:        {final_results.get('training_time_minutes', 'N/A'):.2f} minutes")
    
    # Per-hospital results
    if 'hospitals' in final_results:
        print("\n  Per-Hospital Metrics:")
        for hospital_id, metrics in final_results['hospitals'].items():
            print(f"\n    Hospital {hospital_id}:")
            print(f"      Accuracy:  {metrics.get('accuracy', 'N/A'):.4f}")
            print(f"      F1 Macro:  {metrics.get('f1_macro', 'N/A'):.4f}")
            print(f"      Epsilon:   {metrics.get('epsilon', 'N/A'):.4f}")
else:
    print("\n⚠️  Final results file not found")

print(f"\n\n📁 All results saved to: {results_dir}/")
print("="*70)

## Cell 10: Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results_dir = Path(cfg.output_root) / "results"

# 1. Privacy Budget Over Rounds
privacy_file = results_dir / "privacy_budget.csv"
if privacy_file.exists():
    privacy_df = pd.read_csv(privacy_file)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Privacy budget consumption
    for hospital in privacy_df['hospital_id'].unique():
        hospital_data = privacy_df[privacy_df['hospital_id'] == hospital]
        ax1.plot(hospital_data['round'], hospital_data['epsilon'], marker='o', label=f'Hospital {hospital}')
    
    ax1.axhline(y=cfg.target_epsilon, color='r', linestyle='--', label=f'Target ε={cfg.target_epsilon}')
    ax1.set_xlabel('Round')
    ax1.set_ylabel('ε (Epsilon)')
    ax1.set_title('Privacy Budget Consumption Over Rounds')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Budget remaining
    for hospital in privacy_df['hospital_id'].unique():
        hospital_data = privacy_df[privacy_df['hospital_id'] == hospital]
        ax2.plot(hospital_data['round'], hospital_data['budget_remaining'], marker='s', label=f'Hospital {hospital}')
    
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Budget Remaining')
    ax2.set_title('Privacy Budget Remaining Over Rounds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'privacy_budget_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Privacy budget visualization saved")

# 2. Adaptive Weights Over Rounds
weights_file = results_dir / "adaptive_weights.csv"
if weights_file.exists():
    weights_df = pd.read_csv(weights_file)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Weights over rounds
    for hospital in weights_df['hospital_id'].unique():
        hospital_data = weights_df[weights_df['hospital_id'] == hospital]
        ax1.plot(hospital_data['round'], hospital_data['weight'], marker='o', label=f'Hospital {hospital}')
    
    ax1.set_xlabel('Round')
    ax1.set_ylabel('Aggregation Weight')
    ax1.set_title('Adaptive Aggregation Weights Over Rounds')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Composite scores
    for hospital in weights_df['hospital_id'].unique():
        hospital_data = weights_df[weights_df['hospital_id'] == hospital]
        ax2.plot(hospital_data['round'], hospital_data['score'], marker='s', label=f'Hospital {hospital}')
    
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Composite Score')
    ax2.set_title('Hospital Composite Scores Over Rounds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'adaptive_weights_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Adaptive weights visualization saved")

## Cell 11: Export Results

In [ ]:
import shutil

results_dir = Path(cfg.output_root) / "results"

# Create results directory if it doesn't exist
results_dir.mkdir(parents=True, exist_ok=True)

# Create a summary file
summary_file = results_dir / "KAGGLE_TRAINING_SUMMARY.txt"

with open(summary_file, 'w') as f:
    f.write("="*70 + "\n")
    f.write("HPAFL Training Summary - Kaggle T4 GPU\n")
    f.write("="*70 + "\n\n")
    
    f.write("CONFIGURATION\n")
    f.write("-" * 70 + "\n")
    f.write(f"Rounds: {cfg.num_rounds}\n")
    f.write(f"Local Epochs: {cfg.local_epochs}\n")
    f.write(f"Batch Size: {cfg.batch_size}\n")
    f.write(f"Model: {cfg.model_name}\n")
    f.write(f"Image Size: {cfg.image_size}\n")
    f.write(f"Num Hospitals: {cfg.num_hospitals}\n\n")
    
    f.write("PRIVACY PARAMETERS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Target Epsilon: {cfg.target_epsilon}\n")
    f.write(f"Target Delta: {cfg.target_delta}\n")
    f.write(f"Max Grad Norm: {cfg.max_grad_norm}\n")
    f.write(f"Noise Multiplier: {cfg.noise_multiplier}\n\n")
    
    f.write("OUTPUT FILES\n")
    f.write("-" * 70 + "\n")
    for file in sorted(results_dir.glob('*')):
        if file.is_file():
            size_mb = file.stat().st_size / 1e6
            f.write(f"  {file.name} ({size_mb:.2f} MB)\n")

print(f"\n✓ Summary saved to {summary_file}")

# List all output files
print("\n" + "="*70)
print("OUTPUT FILES")
print("="*70)
for file in sorted(results_dir.glob('*')):
    if file.is_file():
        size_mb = file.stat().st_size / 1e6
        print(f"  {file.name:<40} {size_mb:>8.2f} MB")